In [535]:
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional

In [536]:

def estadisticas_basicas(precios: pd.Series) -> Dict:
    """
    Calcula estadísticas descriptivas de los precios.
    """
    resultado = {}
    resultado["precio_actual"] = precios.iloc[-1]
    resultado["precio_minimo"] = precios.min()
    resultado["precio_maximo"] = precios.max()
    resultado["precio_promedio"] = precios.mean()
    resultado["precio_mediana"] = precios.median()
    resultado["desviacion_std"] = precios.std()
    resultado["rango"] = precios.max() - precios.min()
    resultado["dias_analizados"] = len(precios)

    
    return resultado
    
    # TODO: Implementar usando métodos de Series
    # Tip: .iloc[-1] para el último valor, .min(), .max(), etc.
    


In [537]:
def calcular_rendimientos(precios: pd.Series) -> pd.Series:
    """
    Calcula el rendimiento diario porcentual.
    """
    # TODO: Implementar
    # Tip: Usa .pct_change() y multiplica por 100
    return precios.pct_change() * 100
    pass

In [538]:
def analisis_rendimientos(rendimientos: pd.Series) -> Dict:

    resultado = {}

    resultado["rendimiento_total"] = rendimientos.sum()
    resultado["rendimiento_promedio"] = rendimientos.mean()

    resultado["mejor_dia"] = (
        rendimientos.idxmax(),
        rendimientos.max()
    )

    resultado["peor_dia"] = (
        rendimientos.idxmin(),
        rendimientos.min()
    )

    resultado["dias_positivos"] = (rendimientos > 0).sum()
    resultado["dias_negativos"] = (rendimientos < 0).sum()
    resultado["volatilidad"] = rendimientos.std()

    return resultado
    
     

In [539]:
# PARTE 2: Indicadores Técnicos

def media_movil(precios: pd.Series, ventana: int) -> pd.Series:
    """
    Calcula la media móvil simple (SMA).
    """
    return precios.rolling(window=ventana).mean()
    # TODO: Implementar
    # Tip: Usa .rolling(window=ventana).mean()
    pass

In [540]:
def bandas_bollinger(precios: pd.Series, ventana: int = 20, num_std: int = 2) -> Dict:
    """
    Calcula las Bandas de Bollinger.
    """
    resultado = {}
    media = precios.rolling(window=ventana).mean()
    std = precios.rolling(window=ventana).std()

    resultado["banda_media"] = media
    resultado["banda_superior"] = media + num_std * std
    resultado["banda_inferior"] = media - num_std * std


    
    
    # TODO: Implementar
    # Tip: .rolling(window).mean() y .rolling(window).std()
    
    return resultado

In [541]:
def detectar_maximos_minimos(precios: pd.Series, ventana: int = 5) -> Dict:
    """
    Detecta máximos y mínimos locales.
    """
    resultado = {}
    max_local = precios[precios == precios.rolling(window=ventana, center=True).max()]
    min_local = precios[precios == precios.rolling(window=ventana, center=True).min()]

    resultado["maximos"] = max_local
    resultado["minimos"] = min_local

    return resultado
    
    
    # TODO: Implementar
    # Tip: Compara cada precio con .rolling(window).max() y .min()
    


In [542]:
def clasificar_tendencia(precios: pd.Series, ventana: int = 10) -> str:
    """
    Clasifica la tendencia actual.
    
    """
    # TODO: Implementar
    # 1. Calcular MA
    # 2. Comparar precio actual vs MA
    # 3. Verificar si MA está subiendo o bajando
    
    ma = media_movil(precios, ventana)

    precio_actual = precios.iloc[-1]
    ma_actual = ma.iloc[-1]
    ma_anterior = ma.iloc[-2]

    if precio_actual > ma_actual and ma_actual > ma_anterior:
      return "ALCISTA"

    elif precio_actual < ma_actual and ma_actual < ma_anterior:
      return "BAJISTA"
    else:
       return "LATERAL"
    pass
     

In [543]:
def generar_senales_trading(precios: pd.Series, ma_corta: int = 5, ma_larga: int = 20) -> pd.Series:
    """
    Genera señales de compra/venta basadas en cruces de medias móviles.
    """

    ma_c = media_movil(precios, ma_corta)
    ma_l = media_movil(precios, ma_larga)

    senales = pd.Series(index=precios.index, dtype="object")
    senales[:] = "MANTENER"

    for i in range(1, len(precios)):

        if pd.isna(ma_c.iloc[i-1]) or pd.isna(ma_l.iloc[i-1]):
            continue

        if ma_c.iloc[i-1] <= ma_l.iloc[i-1] and ma_c.iloc[i] > ma_l.iloc[i]:
            senales.iloc[i] = "COMPRA"

        elif ma_c.iloc[i-1] >= ma_l.iloc[i-1] and ma_c.iloc[i] < ma_l.iloc[i]:
            senales.iloc[i] = "VENTA"

    return senales

In [544]:
def alertas_precio(precios: pd.Series, umbral_cambio: float = 5.0) -> List[Dict]:
    """
    Genera alertas cuando hay cambios significativos.
    """
    alertas = []
    rendimientos = calcular_rendimientos(precios)

    for fecha, cambio in rendimientos.items():
    
     if abs(cambio) > umbral_cambio:
            alertas.append({
            "fecha": fecha,
            "cambio_porcentual": cambio,
            "tipo": "SUBIDA" if cambio > 0 else "CAIDA"
        })

    return alertas
    
    # TODO: Implementar
    # 1. Calcular rendimientos
    # 2. Filtrar donde |rendimiento| > umbral
    # 3. Crear lista de alertas
    
    return alertas

In [545]:
def clasificar_volatilidad(rendimientos: pd.Series) -> str:
    """
    Clasifica el nivel de volatilidad del activo.
    """
    # TODO: Implementar
    # Calcular desviación estándar y clasificar
    vol = rendimientos.std()

    if vol < 1:
     return "BAJA"
    elif vol < 3:
     return "MEDIA"
    else:
     return "ALTA"
    pass

In [546]:
def generar_reporte_completo(precios: pd.Series, nombre_accion: str) -> Dict:
    """
    Genera un reporte completo de análisis.
    """

    reporte = {}

    rendimientos = calcular_rendimientos(precios)

    reporte["nombre_accion"] = nombre_accion

    reporte["periodo"] = {
        "inicio": precios.index[0],
        "fin": precios.index[-1]
    }

    reporte["estadisticas"] = estadisticas_basicas(precios)
    reporte["rendimientos"] = analisis_rendimientos(rendimientos)
    reporte["tendencia"] = clasificar_tendencia(precios)
    reporte["volatilidad"] = clasificar_volatilidad(rendimientos)

    senales = generar_senales_trading(precios)
    reporte["senal_actual"] = senales.iloc[-1]

    reporte["alertas_recientes"] = alertas_precio(precios)

    return reporte

In [547]:
# Simular 60 días de precios de una acción
np.random.seed(42)  # Para reproducibilidad

# Generar fechas
fechas = pd.date_range(start='2024-01-01', periods=60, freq='B')  # B = días hábiles

# Generar precios con tendencia alcista y volatilidad
precio_inicial = 100
rendimientos_simulados = np.random.normal(0.002, 0.02, 60)  # Media 0.2%, std 2%
precios_simulados = precio_inicial * np.cumprod(1 + rendimientos_simulados)

# Crear Serie de precios
PRECIOS_ACCION = pd.Series(
    precios_simulados.round(2),
    index=fechas,
    name='ACME Corp'
)

print("Precios de ACME Corp (primeros 10 días):")
print(PRECIOS_ACCION.head(10))
print(f"\nTotal de días: {len(PRECIOS_ACCION)}")

Precios de ACME Corp (primeros 10 días):
2024-01-01    101.19
2024-01-02    101.12
2024-01-03    102.63
2024-01-04    105.96
2024-01-05    105.68
2024-01-08    105.39
2024-01-09    108.93
2024-01-10    110.82
2024-01-11    110.00
2024-01-12    111.42
Freq: B, Name: ACME Corp, dtype: float64

Total de días: 60


In [548]:
# Datos adicionales para pruebas más completas
np.random.seed(123)

# Acción con alta volatilidad
rend_volatil = np.random.normal(0, 0.05, 60)  # 5% de volatilidad diaria
precios_volatil = 50 * np.cumprod(1 + rend_volatil)
ACCION_VOLATIL = pd.Series(
    precios_volatil.round(2),
    index=fechas,
    name='VolatilTech'
)

# Acción con tendencia bajista
rend_bajista = np.random.normal(-0.005, 0.015, 60)  # Tendencia negativa
precios_bajista = 200 * np.cumprod(1 + rend_bajista)
ACCION_BAJISTA = pd.Series(
    precios_bajista.round(2),
    index=fechas,
    name='DeclineCorp'
)

print("Acciones disponibles para análisis:")
print(f"1. ACME Corp - Precio actual: ${PRECIOS_ACCION.iloc[-1]:.2f}")
print(f"2. VolatilTech - Precio actual: ${ACCION_VOLATIL.iloc[-1]:.2f}")
print(f"3. DeclineCorp - Precio actual: ${ACCION_BAJISTA.iloc[-1]:.2f}")

Acciones disponibles para análisis:
1. ACME Corp - Precio actual: $92.74
2. VolatilTech - Precio actual: $59.42
3. DeclineCorp - Precio actual: $138.97


In [549]:
def mostrar_reporte(reporte: Dict) -> None:
    """Muestra el reporte de forma legible."""
    print("=" * 70)
    print(f"           REPORTE DE ANÁLISIS: {reporte['nombre_accion']}")
    print("=" * 70)
    
    # Período
    periodo = reporte.get('periodo', {})
    print(f"\n📅 PERÍODO DE ANÁLISIS")
    print("-" * 40)
    print(f"Inicio: {periodo.get('inicio', 'N/A')}")
    print(f"Fin: {periodo.get('fin', 'N/A')}")
    print(f"Días analizados: {reporte['estadisticas']['dias_analizados']}")
    
    # Estadísticas
    stats = reporte.get('estadisticas', {})
    print(f"\n📊 ESTADÍSTICAS DE PRECIO")
    print("-" * 40)
    print(f"Precio actual:  ${stats.get('precio_actual', 0):,.2f}")
    print(f"Precio mínimo:  ${stats.get('precio_minimo', 0):,.2f}")
    print(f"Precio máximo:  ${stats.get('precio_maximo', 0):,.2f}")
    print(f"Precio promedio: ${stats.get('precio_promedio', 0):,.2f}")
    
    # Rendimientos
    rend = reporte.get('rendimientos', {})
    print(f"\n📈 RENDIMIENTO")
    print("-" * 40)
    print(f"Rendimiento total: {rend.get('rendimiento_total', 0):+.2f}%")
    print(f"Rendimiento promedio diario: {rend.get('rendimiento_promedio', 0):+.3f}%")
    if rend.get('mejor_dia'):
        print(f"Mejor día: {rend['mejor_dia'][0]} ({rend['mejor_dia'][1]:+.2f}%)")
    if rend.get('peor_dia'):
        print(f"Peor día: {rend['peor_dia'][0]} ({rend['peor_dia'][1]:+.2f}%)")
    print(f"Días positivos: {rend.get('dias_positivos', 0)}")
    print(f"Días negativos: {rend.get('dias_negativos', 0)}")
    
    # Indicadores
    print(f"\n🎯 INDICADORES")
    print("-" * 40)
    print(f"Tendencia: {reporte.get('tendencia', 'N/A')}")
    print(f"Volatilidad: {reporte.get('volatilidad', 'N/A')}")
    print(f"Señal actual: {reporte.get('senal_actual', 'N/A')}")
    
    # Alertas
    alertas = reporte.get('alertas_recientes', [])
    if alertas:
        print(f"\n⚠️ ALERTAS RECIENTES")
        print("-" * 40)
        for alerta in alertas[-5:]:  # Últimas 5
            emoji = "🔺" if alerta['tipo'] == 'SUBIDA' else "🔻"
            print(f"{emoji} {alerta['fecha']}: {alerta['tipo']} de {alerta['cambio_porcentual']:+.2f}%")
    
    print("\n" + "=" * 70)

In [550]:
def visualizar_precios_texto(precios: pd.Series, ancho: int = 50) -> None:
    """Visualización simple de precios en texto (ASCII chart)."""
    min_precio = precios.min()
    max_precio = precios.max()
    rango = max_precio - min_precio
    
    print(f"\nGráfico de precios: {precios.name}")
    print(f"Max: ${max_precio:.2f}")
    print("-" * (ancho + 10))
    
    # Mostrar cada 3 días para no saturar
    for fecha, precio in precios.iloc[::3].items():
        posicion = int((precio - min_precio) / rango * ancho) if rango > 0 else ancho // 2
        barra = " " * posicion + "█"
        fecha_str = fecha.strftime('%m/%d') if hasattr(fecha, 'strftime') else str(fecha)[:5]
        print(f"{fecha_str} |{barra}")
    
    print("-" * (ancho + 10))
    print(f"Min: ${min_precio:.2f}")

In [551]:
# Prueba de funciones individuales
print("PRUEBA DE FUNCIONES INDIVIDUALES")
print("=" * 50)

# Estadísticas básicas
print("\n-- Estadísticas Básicas --")
stats = estadisticas_basicas(PRECIOS_ACCION)
print(stats)

# Rendimientos
print("\n-- Rendimientos (primeros 5) --")
rendimientos = calcular_rendimientos(PRECIOS_ACCION)
print(rendimientos.head())

# Análisis de rendimientos
print("\n-- Análisis de Rendimientos --")
analisis = analisis_rendimientos(rendimientos)
print(analisis)

PRUEBA DE FUNCIONES INDIVIDUALES

-- Estadísticas Básicas --
{'precio_actual': np.float64(92.74), 'precio_minimo': np.float64(86.67), 'precio_maximo': np.float64(111.42), 'precio_promedio': np.float64(96.88983333333334), 'precio_mediana': np.float64(95.645), 'desviacion_std': np.float64(7.128231646939469), 'rango': np.float64(24.75), 'dias_analizados': 60}

-- Rendimientos (primeros 5) --
2024-01-01         NaN
2024-01-02   -0.069177
2024-01-03    1.493275
2024-01-04    3.244665
2024-01-05   -0.264251
Freq: B, Name: ACME Corp, dtype: float64

-- Análisis de Rendimientos --
{'rendimiento_total': np.float64(-7.748229086684423), 'rendimiento_promedio': np.float64(-0.1313259167234648), 'mejor_dia': (Timestamp('2024-02-13 00:00:00'), np.float64(3.905831995719633)), 'peor_dia': (Timestamp('2024-02-21 00:00:00'), np.float64(-3.714554776603529)), 'dias_positivos': np.int64(26), 'dias_negativos': np.int64(33), 'volatilidad': np.float64(1.823543256771936)}


In [552]:
# Prueba de indicadores técnicos
print("\n-- Media Móvil (5 días) --")
ma5 = media_movil(PRECIOS_ACCION, 5)
print(ma5.tail())

print("\n-- Bandas de Bollinger --")
bandas = bandas_bollinger(PRECIOS_ACCION, 20, 2)
for nombre, serie in bandas.items():
    if serie is not None:
        print(f"{nombre}: {serie.iloc[-1]:.2f}")

print("\n-- Tendencia --")
tendencia = clasificar_tendencia(PRECIOS_ACCION)
print(f"Tendencia actual: {tendencia}")


-- Media Móvil (5 días) --
2024-03-18    88.776
2024-03-19    89.318
2024-03-20    89.986
2024-03-21    90.564
2024-03-22    91.134
Freq: B, Name: ACME Corp, dtype: float64

-- Bandas de Bollinger --
banda_media: 89.85
banda_superior: 93.65
banda_inferior: 86.06

-- Tendencia --
Tendencia actual: ALCISTA


In [553]:
# Prueba del reporte completo
print("\nGENERANDO REPORTE COMPLETO...\n")
reporte = generar_reporte_completo(PRECIOS_ACCION, "ACME Corp")
print(f"           REPORTE DE ANÁLISIS: {reporte['nombre_accion']}")


GENERANDO REPORTE COMPLETO...

           REPORTE DE ANÁLISIS: ACME Corp


In [554]:
# Comparar las tres acciones
print("\n" + "=" * 70)
print("         COMPARACIÓN DE ACCIONES")
print("=" * 70)

acciones = [
    (PRECIOS_ACCION, "ACME Corp"),
    (ACCION_VOLATIL, "VolatilTech"),
    (ACCION_BAJISTA, "DeclineCorp")
]

for precios, nombre in acciones:
    rendimientos = calcular_rendimientos(precios)
    if rendimientos is not None:
        rend_total = rendimientos.sum() if not rendimientos.isna().all() else 0
        volatilidad = clasificar_volatilidad(rendimientos)
        tendencia = clasificar_tendencia(precios)
        
        print(f"\n{nombre}:")
        print(f"  Rendimiento: {rend_total:+.2f}%")
        print(f"  Volatilidad: {volatilidad}")
        print(f"  Tendencia: {tendencia}")


         COMPARACIÓN DE ACCIONES

ACME Corp:
  Rendimiento: -7.75%
  Volatilidad: MEDIA
  Tendencia: ALCISTA

VolatilTech:
  Rendimiento: +33.35%
  Volatilidad: ALTA
  Tendencia: ALCISTA

DeclineCorp:
  Rendimiento: -33.81%
  Volatilidad: MEDIA
  Tendencia: BAJISTA


In [555]:
print(reporte.keys())

dict_keys(['nombre_accion', 'periodo', 'estadisticas', 'rendimientos', 'tendencia', 'volatilidad', 'senal_actual', 'alertas_recientes'])


In [556]:
print(reporte)

{'nombre_accion': 'ACME Corp', 'periodo': {'inicio': Timestamp('2024-01-01 00:00:00'), 'fin': Timestamp('2024-03-22 00:00:00')}, 'estadisticas': {'precio_actual': np.float64(92.74), 'precio_minimo': np.float64(86.67), 'precio_maximo': np.float64(111.42), 'precio_promedio': np.float64(96.88983333333334), 'precio_mediana': np.float64(95.645), 'desviacion_std': np.float64(7.128231646939469), 'rango': np.float64(24.75), 'dias_analizados': 60}, 'rendimientos': {'rendimiento_total': np.float64(-7.748229086684423), 'rendimiento_promedio': np.float64(-0.1313259167234648), 'mejor_dia': (Timestamp('2024-02-13 00:00:00'), np.float64(3.905831995719633)), 'peor_dia': (Timestamp('2024-02-21 00:00:00'), np.float64(-3.714554776603529)), 'dias_positivos': np.int64(26), 'dias_negativos': np.int64(33), 'volatilidad': np.float64(1.823543256771936)}, 'tendencia': 'ALCISTA', 'volatilidad': 'MEDIA', 'senal_actual': 'MANTENER', 'alertas_recientes': []}


In [ ]:
reporte = generar_reporte_completo(PRECIOS_ACCION, "ACME Corp")
mostrar_reporte(reporte) 

           REPORTE DE ANÁLISIS: ACME Corp

📅 PERÍODO DE ANÁLISIS
----------------------------------------
Inicio: 2024-01-01 00:00:00
Fin: 2024-03-22 00:00:00
Días analizados: 60

📊 ESTADÍSTICAS DE PRECIO
----------------------------------------
Precio actual:  $92.74
Precio mínimo:  $86.67
Precio máximo:  $111.42
Precio promedio: $96.89

📈 RENDIMIENTO
----------------------------------------
Rendimiento total: -7.75%
Rendimiento promedio diario: -0.131%
Mejor día: 2024-02-13 00:00:00 (+3.91%)
Peor día: 2024-02-21 00:00:00 (-3.71%)
Días positivos: 26
Días negativos: 33

🎯 INDICADORES
----------------------------------------
Tendencia: ALCISTA
Volatilidad: MEDIA
Señal actual: MANTENER

